## Step 0: Overview

#### 1. Agent 소개
이 노트북을 통해, **초등학교 3학년 학생의 수학선생님 Agent**을 만들어 보도록 합니다.

#### 2. Agent의 주요 특징 및 목표
- Episode 형식의 메모리를 통하여, 학생의 이전 주차 학습 진도를 지속적으로 트래킹 할 수 있습니다
- Reflection 메모리를 통하여, 과거 학습 세션에서 얻은 인사이트들을 기억합니다.(학생의 성향이나 이해도와 같은 정보들)

#### 3. 기대하는 테스트 시나리오 
- ex. 선생님 지난번 정수의 곱셈 문제를 풀때 헷갈렸던 개념이 있었는데... 다시 알려주실 수 있나요?
- ex. 선생님 제가 공부를 할때 고쳐야 하는 점이나 잘못된 습관이 있나요?

#### 4. 동작 방식
![0-agent-overview](./figures/0-agent-overview.png)

#### 5. 개발 과정 및 순서
![00-process-overview](./figures/00-process-overview.png)

## Step 1: Install Dependencies and Setup
필요한 패키지들을 설정하고, Memory 생성을 위한 설정들을 해줍니다.

In [ ]:
%pip install -qr requirements.txt

In [ ]:
import json
import logging
import sys
import uuid
from datetime import datetime, timezone
from typing import List, Dict, Any
from pprint import pprint
import os
import time

# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("math-teacher-assistant")

# Import boto3 for control plane and data plane operations
import boto3

# Import Strands Agent framework
from strands import Agent, tool

logger.info("✅ All dependencies imported successfully")

In [ ]:
# Configuration
REGION = 'us-east-1'
actor_id = "student_01_minsu"
memory_name = "MathTeacherAgent"

logger.info(f"Configuration set for region: {REGION}")
logger.info(f"Actor ID: {actor_id}")

In [ ]:
import json

# Get current AWS account ID
sts = boto3.client('sts')
account_id = sts.get_caller_identity()['Account']

# IAM role name
role_name = 'BedrockAgentCoreMemoryExecutionRole'

# Create IAM client
iam = boto3.client('iam')

try:
    # Try to get existing role
    response = iam.get_role(RoleName=role_name)
    memory_execution_role_arn = response['Role']['Arn']
    logger.info(f"✅ Using existing role: {memory_execution_role_arn}")
    
except iam.exceptions.NoSuchEntityException:
    # Role doesn't exist, create it
    logger.info(f"Creating new role: {role_name}")
    
    # Trust policy
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {"aws:SourceAccount": account_id},
                "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:*"}
            }
        }]
    }
    
    # Create role
    response = iam.create_role(
        RoleName=role_name,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='Execution role for Bedrock AgentCore Memory'
    )
    memory_execution_role_arn = response['Role']['Arn']
    
    # Attach permissions policy
    permissions_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "bedrock:InvokeModel",
                    "bedrock:InvokeModelWithResponseStream"
                ],
                "Resource": [
                    "arn:aws:bedrock:*::foundation-model/*",
                    "arn:aws:bedrock:*:*:inference-profile/*"
                ]
            },
            {
                "Effect": "Allow",
                "Action": [
                    "aws-marketplace:ViewSubscriptions",
                    "aws-marketplace:Subscribe"
                ],
                "Resource": "*"
            }
        ]
    }
    
    iam.put_role_policy(
        RoleName=role_name,
        PolicyName='BedrockModelInvokePolicy',
        PolicyDocument=json.dumps(permissions_policy)
    )
    
    logger.info(f"✅ Created role: {memory_execution_role_arn}")
    logger.info("Waiting 10 seconds for IAM propagation...")
    import time
    time.sleep(10)


## Step 2: Create Memory with Episodic Strategy
![1-create-memory](./figures/1-create-memory.png)

다음으로, Episodic Strategy을 활용한 Bedrock Agent Core Memory을 생성합니다.
- 수업을 진행한 세션 정보를 Episode 형태로 저장합니다.
- 관련성이 높은 Episodes들을 모아 Insight을 생성하여 Reflection Memory에 기록합니다.

![01-create-episodic-strategy](./figures/01-create-episodic-strategy.png)

Agentcore Memory을 생성하기에 앞서, Memory에 사용할 Strategy을 정의해주어야 합니다.

이때 Strategy을 구성하는 항목들은 아래와 같습니다 : 

1. Prompts (Optional)
    
    Built-in Strategy을 이용하여 기본적으로 제공되는 Prompt을 사용할 수 있으며, 기본 제공 Prompt에 지시사항을 추가적으로 Override하고 싶은 경우, Override Strategy을 이용합니다. 이는 기본 제공 Prompt에 Custom Prompt가 Append 되는 방식입니다.

    - Episode Extraction Prompt
    - Episode Consolidation Prompt
    - Reflection Prompt

2. Namespace
    
    Agentcore Memory는 Namespace 구조를 사용하여 마치 파일시스템 처럼 데이터를 논리적으로 구분하고 격리합니다. 이는 Retrieve 혹은 Filtering 과정에서 이용되며, Access Control을 위한 격리에도 이용될 수 있습니다.

    ![namespace-structure](./figures/namespace-structure.png)

    에피소드는 아래의 Namespace 중 하나에 저장될 수 있습니다 : 
    - `/strategy/{memoryStrategyId}/` : Strategy 레벨에서 에피소드를 저장합니다. 다른 Actor ID나 Session ID을 가지더라도, 같은 Strategy을 사용할 경우 같은 Namespace 아래에 저장됩니다.
    - `/strategy/{memoryStrategyId}/actor/{actorId}/` : Actor 레벨에서 에피소드를 저장합니다. 다른 Session ID을 가지더라도, 같은 Actor ID이라면 같은 Namespace 아래에 저장됩니다.
    - `/strategy/{memoryStrategyId}/actor/{actorId}/session/{sessionId}/` : Session 레벨에서 에피소드를 저장합니다. 같은 Session에 해당하는 에피소드만 같은 Namespace 아래에 저장됩니다.

    Reflection은 아래의 Namespace 중 하나에 저장될 수 있습니다 : 
    - `/strategy/{memoryStrategyId}/actor/{actorId}/` : Actor 레벨에서 Reflection을 저장합니다. 한 Actor의 여러 세션에 걸쳐 Reflection이 생성되며 저장됩니다.
    - `/strategy/{memoryStrategyId}/` : 모든 Actor에 걸쳐 Reflection이 생성되며 저장됩니다.

3. LLM Models
    
    Episode Extraction, Consolidation, Reflection 단계에서 사용할 LLM Model을 선택합니다.

4. Execution Role (Optional)

    Built-in Override 방식을 사용할때에는 CreateMemory을 하기 위해 memoryExecutionRoleARN을 반드시 전달해야 합니다. Bedrock AgentCore는 이 역할을 Assume하여 고객 계정의 Bedrock 모델을 사용해 Extraction, Consolidation을 수행합니다.

In [ ]:
# Initialize boto3 client for control and data plane operations
client = boto3.client(
    'bedrock-agentcore',
    region_name=REGION,
)
memory_client = boto3.client(
  'bedrock-agentcore-control',
   region_name=REGION,
)

기본적인 Built-in Strategy을 이용하는 Episodic Strategy 방식은 아래와 같습니다.

```json
episodic_strategy = {
    "episodicMemoryStrategy": {
      "name": "MathLearningEpisodeExtractor",
      "description": "Creates Math session episodes with reflections per actor",
      "namespaces": [
        "math_learning/{actorId}/sessions/{sessionId}"
      ],
      "reflectionConfiguration": {
        "namespaces": [
          "math_learning/{actorId}" 
        ]
      }
    }
}
```

하지만 본 노트북에서는 Built-in with Override Strategy 방식을 이용하여 진행합니다.
Built-in with Override Strategy을 사용하기 위해서는 아래 사항들을 추가로 설정해주어야 합니다.
- `EXTRACTION_PROMPT` 
- `CONSOLIDATION_PROMPT`
- `REFLECTION_PROMPT`
- Execution Role

> 📌 참고 : 본 노트북에서는 `EXTRACTION_PROMPT`만을 Override하여 Strategy을 구성하였습니다.

In [ ]:
EXTRACTION_PROMPT = """
## Math Tutoring Episode Extraction Rules

This is a conversation between a math teacher and an elementary student.

### Curriculum (진도표)
1주차: 정수의 곱셈
2주차: 정수의 나눗셈
3주차: 분수
4주차: 소수
5주차: 길이 단위 (cm, m, km)
6주차: 시간 단위 (초, 분, 시)
7주차: 무게 단위 (g, kg)
8주차: 들이 단위 (mL, L)

### Extraction Guidelines

For <situation>:
- Use the curriculum above to identify the current week based on the topic being taught
- Include current week number and topic (e.g., "3주차 분수 수업")
- Note if this is 복습 or 새 진도

For <action>:
- Include the specific problem and student's answer
- Describe what happened: problem → student's solving steps → answer (correct/incorrect)
- Example: "23×4 문제 → 3×4=12, 2×4=8로 계산 → 82 답함 (오답, 정답: 92)"

For <thought>:
- Why the error occurred (underlying misconception)
- Teaching method used and why
- Any patterns observed in student's behavior

### Episode Boundary
- End the episode when a math problem is completed (correct or incorrect), and start a new episode for the next problem.

"""

CONSOLIDATION_PROMPT = "Consolidate"
REFLECTION_PROMPT = "Reflect"

In [ ]:
custom_episodic_strategy = {
    "customMemoryStrategy": {
        "name": "MathLearningEpisodeExtractor",
        "description": "수학 학습 세션에서 학생의 실수와 교정 과정을 추출",
        "namespaces": [
            "math_learning/{actorId}/sessions/{sessionId}" # Session Level에 Episode 저장
        ],
        "configuration": {
            "episodicOverride": {
                "extraction": {
                    "appendToPrompt": EXTRACTION_PROMPT,
                    "modelId": "us.anthropic.claude-sonnet-4-20250514-v1:0"
                },
                "consolidation": {
                    "appendToPrompt": CONSOLIDATION_PROMPT,
                    "modelId": "us.anthropic.claude-sonnet-4-20250514-v1:0"  
                },
                "reflection": {
                    "namespaces": [
                        "math_learning/{actorId}" # Actor Level에 Reflection 저장
                    ],
                    "appendToPrompt": REFLECTION_PROMPT, 
                    "modelId": "us.anthropic.claude-sonnet-4-20250514-v1:0"  
                }
            }
        }
    }
}



다음으로, Execution Role을 생성하여 `create_memory` 단계에서 전달될 수 있도록 합니다.

```json
# Trust policy
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {"aws:SourceAccount": account_id},
                "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:*"}
            }
        }]
    }
    
# Attach permissions policy
permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "bedrock:InvokeModel",
                "bedrock:InvokeModelWithResponseStream"
            ],
            "Resource": [
                "arn:aws:bedrock:*::foundation-model/*",
                "arn:aws:bedrock:*:*:inference-profile/*"
            ]
        },
        {
            "Effect": "Allow",
            "Action": [
                "aws-marketplace:ViewSubscriptions",
                "aws-marketplace:Subscribe"
            ],
            "Resource": "*"
        }
    ]
}

```

![02-create-memory](./figures/02-create-memory.png)

위에서 구성한 Memory Strategy 설정사항들을 바탕으로 Memory을 생성합니다. 
```python
response = memory_client.create_memory(
            name=memory_name,
            description="초등학교 3학년 수학 학습 세션을 관리합니다",
            eventExpiryDuration=90,
            memoryStrategies=[custom_episodic_strategy],
            memoryExecutionRoleArn= memory_execution_role_arn,
            clientToken=str(uuid.uuid4())
        )
```

In [ ]:
# Get or create memory
try:
    # Try to find existing memory first
    list_response = memory_client.list_memories(maxResults=100)
    memory_id = None
    for mem in list_response.get('memories', []):
        detail = memory_client.get_memory(memoryId=mem['id'])
        if detail['memory'].get('name') == memory_name:
            memory_id = mem['id']
            logger.info(f"Using existing memory: {memory_id}")
            break
    
    # Create if not found
    if not memory_id:
        logger.info(f"Creating new memory: {memory_name}")
        response = memory_client.create_memory(
            name=memory_name,
            description="초등학교 3학년 수학 학습 세션을 관리합니다",
            eventExpiryDuration=90,
            memoryStrategies=[custom_episodic_strategy],
            memoryExecutionRoleArn= memory_execution_role_arn,
            clientToken=str(uuid.uuid4())
        )
        memory_id = response['memory']['id']
        logger.info(f"Memory created: {memory_id}")
        
        # Wait for ACTIVE
        import time
        for _ in range(30):
            status = memory_client.get_memory(memoryId=memory_id)['memory']['status']
            if status == 'ACTIVE':
                logger.info("Memory is ACTIVE")
                break
            time.sleep(30)
            
except Exception as e:
    logger.error(f"Failed to get/create memory: {e}")
    raise

메모리 생성이 완료되었으면, Cloudwatch Log Delivery 설정을 완료합니다. 이 설정을 통해 추후에 Episode Extraction, Consolidation, Reflection이 발생하는 절차를 Cloudwatch Log을 통해 추적할 수 있습니다.

In [ ]:
# CloudWatch Log Delivery 설정
try:
    logs_client = boto3.client('logs', region_name=REGION)
    memory_arn = f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:memory/{memory_id}"
    
    # 1. Memory의 delivery source 찾기
    sources = logs_client.describe_delivery_sources()
    source_name = None
    
    for source in sources.get('deliverySources', []):
        if memory_arn in source.get('resourceArns', []):
            source_name = source['name']
            logger.info(f"Found existing delivery source: {source_name}")
            break
    
    # 1-1. Delivery source가 없으면 생성
    if not source_name:
        logger.info("Delivery source not found. Creating new one...")
        
        import random
        source_name = f'bedrock-memory-source-{random.randint(100000, 999999)}'
        
        source_response = logs_client.put_delivery_source(
            name=source_name,
            resourceArn=memory_arn,
            logType='APPLICATION_LOGS',  
            tags={
                'CreatedBy': 'PythonScript',
                'MemoryId': memory_id
            }
        )
        
        logger.info(f"Delivery source created: {source_name}")
    
    # 2. Log Group 생성
    log_group_name = f'/aws/vendedlogs/bedrock-agentcore/memory/APPLICATION_LOGS/{memory_id}'
    try:
        logs_client.create_log_group(logGroupName=log_group_name)
        logger.info(f"Log group created")
    except logs_client.exceptions.ResourceAlreadyExistsException:
        logger.info(f"Log group already exists")
    
    # 3. Delivery Destination 생성
    import random
    destination_name = f'memory-delivery-{random.randint(100000, 999999)}'
    dest_response = logs_client.put_delivery_destination(
        name=destination_name,
        deliveryDestinationConfiguration={
            'destinationResourceArn': f'arn:aws:logs:{REGION}:{account_id}:log-group:{log_group_name}'
        },
        deliveryDestinationType='CWL'
    )
    logger.info(f"Delivery destination created: {destination_name}")
    
    # 4. Delivery 생성
    logs_client.create_delivery(
        deliverySourceName=source_name,
        deliveryDestinationArn=dest_response['deliveryDestination']['arn']
    )
    
    logger.info(f"Log delivery configured! Logs at: {log_group_name}")
    
except Exception as e:
    logger.warning(f"Failed: {e}")
    logger.info(f"Error type: {type(e).__name__}")
    logger.info("Configure manually in console if needed")

print(f"Memory ARN: {memory_arn}")


## Step 3: Ingest Memory with Historical Math Class Sessions
![03-ingest-session-memory](./figures/03-ingest-session-memory.png)

과거의 학습 세션 데이터들을 주입합니다. 이 과정에서 `CreateEvent` API를 호출하여 Short-term Memory의 형태로 세션 데이터를 한번에 저장합니다.

```python
result = client.create_event(
    memoryId=memory_id,
    actorId=actor_id,
    sessionId=session_id,
    eventTimestamp=event_timestamp,
    payload=payload
)
```

데이터는 아래의 요소들이 드러나도록 30~35턴의 대화로 구성하였습니다.
- 1주차 : 정수의 곱셈
    - 세로셈 곱셈, 올림 개념에 대해 배움
    - 급하게 푸느라 올림을 빼먹는 실수
- 2주차 : 정수의 나눗셈 
    - 나눗셈과 나머지에 대해 배움
    - 문제를 대충 읽고 풀어서 오답, 선생님은 천천히 다시 풀어볼 수 있도록 유도
- 3주차 : 분수
    - 분수 덧셈, 분모/분자의 개념에 대해 배움
    - 분수의 덧셈에서 분모를 더해버리는 실수를 하였고, 케이크를 예로 들어 분모는 조각의 갯수라 변하지 않음을 설명함

> 📌 참고 :
>  `./short-data`는 30 ~ 35턴의 데이터로 구성하였으며, `.data`는 조금 더 긴 60 ~ 65턴의 데이터로 구성되어있습니다. 본 예제에서는 시간 관계상 `./short-data`을 이용합니다.

In [ ]:
import os
import glob

# Load all session data files
data_dir = "./short-data"
session_files = sorted(glob.glob(f"{data_dir}/*.json"))

logger.info(f"Found {len(session_files)} session files to ingest")

# Ingest each session
for session_file in session_files:
    session_name = os.path.basename(session_file).replace('.json', '')
    session_id = f"{session_name}_{datetime.now().strftime('%Y%m%d%H%M%S')}"
    
    logger.info(f"Ingesting session: {session_name}")
    
    # Load conversation data
    with open(session_file, 'r') as f:
        conversation = json.load(f)
    
    # Convert to payload format
    payload = []
    for turn in conversation:
        conv_data = turn['conversational']
        payload.append({
            'conversational': {
                'content': {'text': conv_data['content']['text']},
                'role': conv_data['role']
            }
        })
    
    # Create event using boto3 directly
    event_timestamp = datetime.now(timezone.utc)
    result = client.create_event(
        memoryId=memory_id,
        actorId=actor_id,
        sessionId=session_id,
        eventTimestamp=event_timestamp,
        payload=payload
    )
    print(session_id)

    logger.info(f"   ✓ Stored {len(payload)} turns - Event ID: {result['event']['eventId']}")

logger.info(f"✅ Successfully ingested {len(session_files)} math learning sessions")

![04-wait-extraction-module](./figures/04-wait-extraction-module.png)
앞선 단계에서 이전 세션 데이터를 Short-term Memory에 저장하였습니다. 이제 Memory Extraction Module에 의해 Short-term Memory가 Long-term Memory으로 추출되기를 기다립니다. 

> 📌 참고 : 
> 
> 다른 Strategy와 다르게, Episode Strategy의 경우 "Completed Episode"을 인식하였을때 Memory Extraction Module이 시작됩니다. 예를들어 세션 데이터가 추가될때 LLM은 현재 에피소드가 종료되었는지(Goal Achieved)을 판단하고, YES라고 판단된 경우에 한하여 Memory Extraction Module을 시작합니다. 이외에는 15분의 Idle Time을 기다려야 하기 때문에 Long-term Memory가 생성되기까지 시간이 소요될 수 있습니다.

Session Ingestion이 완료될때까지 기다립니다. 이는 Assessment 결과에 따라 최대 15~20분 정도 소요될 수 있습니다.

In [ ]:
namespace = f"math_learning/{actor_id}"
memory_threshold = 5  # 이 개수 이상이면 Ingestion 완료로 간주
max_wait_time = 20 * 60  # 최대 20분 대기
check_interval = 30  # 30초마다 체크

logger.info(f"⏳ Waiting for session ingestion (threshold: {memory_threshold} memories)")
logger.info(f"Max wait time: {max_wait_time/60} minutes, checking every {check_interval} seconds")

start_time = time.time()
for attempt in range(max_wait_time // check_interval):
    elapsed = time.time() - start_time
    
    # Check memory records
    response = client.list_memory_records(
        memoryId=memory_id,
        namespace=namespace,
        maxResults=40
    )
    memories = response.get('memoryRecordSummaries', [])
    
    logger.info(f"[{elapsed/60:.1f} min] Attempt {attempt + 1}: Found {len(memories)} memories")
    
    # Ingestion 완료 체크
    if len(memories) >= memory_threshold:
        logger.info(f"✅ Session ingestion complete! Found {len(memories)} memories after {elapsed/60:.1f} minutes")
        break
    
    # 마지막 시도가 아니면 대기
    if attempt < (max_wait_time // check_interval) - 1:
        time.sleep(check_interval)
else:
    # 타임아웃
    logger.warning(f"⚠️ Ingestion timeout after {max_wait_time/60} minutes. Found {len(memories)} memories (expected >= {memory_threshold})")


## Step 4: (Optional) List Generated Long-term Memory Records
생성된 Episodes와 Reflection 데이터들을 List해보고, Retrieve 해볼 수 있는 부분입니다.

In [ ]:
# 1. 생성된 Memory Records을 살펴봅니다.
namespace = f"math_learning/{actor_id}" 

response = client.list_memory_records(
    memoryId=memory_id,
    namespace=namespace,
    maxResults=40
)
memories = response.get('memoryRecordSummaries', [])
logger.info(f"   Found {len(memories)} memories")


In [ ]:
# 2. Reflection과 Episodes 각각의 갯수 조회 
reflections = []
episodes = []

for memory in memories:
    record_type = memory.get('metadata', {}).get('x-amz-agentcore-memory-recordType', {}).get('stringValue')

    if record_type == 'REFLECTION':
        reflections.append(memory)
    elif record_type == 'BASE':
        episodes.append(memory)

print(f"Episodes: {len(episodes)}")
print(f"Reflections: {len(reflections)}")


## Step 5: Create Agentcore Sesion Manager

![2-create-agent](./figures/2-create-agent.png)

Memory 생성을 완료하였다면, Agent을 생성하고 Memory와 연결하는 단계로 넘어가 봅니다.

![05-create-agent-session-manager](./figures/05-create-agent-session-manager.png)

앞서 생성한 Memory를 Agent와 연결할 준비를 하기 위해 Strands에서 제공하는 `AgentCoreMemorySessionManager`을 이용합니다. 이 매니저를 통해 연동할 메모리(`memory_id`)을 지정하고, 어떤 Namespace에서 Memory을 Retrieve할것인지(`retrieval_config`)을 지정해 줍니다.

![agentcore-session-manager](./figures/agentcore-session-manager.png)

이떄 `AgentCoreMemorySessionManager`은 위와 같이, 적합한 Query에 대한 데이터를 Vector Store으로부터 Retrieve하고, Context에 주입하는 역할을 수행합니다.

In [ ]:
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager

SESSION_ID = "minsu_math_session_week_4" # 새로운 세션 지정
config = AgentCoreMemoryConfig(
    memory_id=memory_id,
    session_id=SESSION_ID,
    actor_id=actor_id,
    retrieval_config={
        f"math_learning/{actor_id}": RetrievalConfig(
            top_k=5,
            relevance_score=0.7
        )
    }
)

session_manager = AgentCoreMemorySessionManager(config, region_name='us-east-1')


## Step 6: Create Teacher Assistant Agent

![06-create-agent](./figures/06-create-agent.png)

`AgentCoreMemorySessionManager`의 생성이 완료되었다면, 이제 Agent을 생성하여 매니저를 연결해줍니다.

In [ ]:
# Create the math teacher agent
math_teacher_agent = Agent(
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    session_manager=session_manager,
    system_prompt = f"""You are an expert Elementary Math Teacher for 3rd grade student 민수.

## 민수의 학습 커리큘럼 (초등 3학년 분수)
1주차: 정수의 곱셈
2주차: 정수의 나눗셈
3주차: 분수
4주차: 소수
5주차: 길이 단위 (cm, m, km)
6주차: 시간 단위 (초, 분, 시)
7주차: 무게 단위 (g, kg)
8주차: 들이 단위 (mL, L)

## Context 활용
You have access to 민수's past learning records in your context.
When reviewing past lessons:
1. Check the curriculum above to identify the topic for that week
2. Search your context for 민수's mistakes from that week
3. Give the EXACT same problem 민수 got wrong before (e.g., "지난번에 1/4를 0.14라고 했었는데, 다시 풀어볼까?")

Always use concrete examples and be warm and encouraging."""
)

logger.info("✅ Math teacher agent created with AgentCore memory integration")


## Step 7: Test the Math Teacher Assistant

![07-test-agent](./figures/07-test-agent.png)

앞서 만들었던 Memory을 연결한 Agent을 성공적으로 완성하였습니다. 이제 처음에 계획하였던 시나리오를 이용해 Agent의 성능을 테스트 해봅니다.

### Test 1: Query for Episode Memory 
- Episode 형태의 메모리를 잘 반환하는지 테스트 합니다. 
- 이때 3주차에서 계획하였던 주요 에피소드들이 잘 드러나도록 설명하는지를 확인합니다.

In [ ]:
query1 = """선생님 안녕하세요! 오늘 4주차 수업 시작해요. 우선 지난 3주차때 배운 내용 복습해주세요."""

logger.info(f"\n{'='*80}")
logger.info(f"Test 1: Query for Episode Memory")
logger.info(f"{'='*80}")
logger.info(f"Query: {query1}\n")

response1 = math_teacher_agent(query1)

저장된 Episode 예시 : 

```json
{
  "situation": "3주차 수업 시작 시점에서 교사가 학생에게 인사하며 분수 학습을 시작하는 상황. 이전 2주간의 나눗셈 학습 기반 위에 새로운 분수 개념을 도입하고자 함.",
  "intent": "3주차 분수 개념을 효과적으로 도입하고, 특히 분수 덧셈의 핵심 원리(분모는 그대로 두고 분자만 더하기)를 학생이 완전히 이해하도록 하는 것이 목표.",
  "assessment": "Yes",
  "justification": "학생이 초기 실수(3/8로 잘못 계산)를 통해 오개념을 드러냈으나, 교사의 체계적인 질문과 구체적 예시를 통해 스스로 오류를 발견하고 수정했으며, 이후 연속적인 문제(3/5, 5/8)에서 정확한 계산을 보여 핵심 원리를 완전히 습득했음을 확인.",
  "reflection": "이 수업에서 가장 효과적이었던 도구는 구체적 시각적 예시(피자, 초콜릿 바, 도넛) 활용과 소크라테스식 질문법이었다. 교사는 추상적인 분수 개념을 실생활 예시로 연결하여 이해도를 높였고, 학생의 오답에 대해 직접적인 정답 제시 대신 논리적 모순을 지적하는 질문(\"피자가 갑자기 8조각이 되었나요?\")을 통해 학생이 스스로 오류를 발견하도록 유도했다. 특히 주목할 점은 오류 활용 학습 전략이다. 학생의 3/8 오답을 즉시 교정하지 않고 구체적 상황으로 돌아가 논리적 사고를 통해 스스로 발견하게 한 것이 핵심이었다. 이후 학생이 제시한 \"피자/케이크를 생각하며 조각 수 확인\"이라는 자기 전략을 즉시 검증할 수 있는 연습 문제를 연속 제시하여 학습을 완전히 정착시켰다. 효과적 패턴: 구체적 예시 → 개념 도입 → 연습 → 오류 발생 → 질문을 통한 자기 발견 → 전략 수립 → 연속 검증. 피해야 할 패턴: 추상적 설명만으로 개념 전달, 오답에 대한 즉시 정답 제시, 학생 주도 전략 수립 없이 교사 중심 해결책 제시.",
  "turns": [(대화 Turn 데이터)...]
}
```

### Test 2: Query for Reflection + Episode Knowledge (Strategic Guidance)
- 에피소드 형식으로 저장한 과거 세션의 정보들과, 그로부터 얻은 Insight을 저장한 Reflection을 결합하여 대답할수 있는지를 확인합니다.
- 예를들어, 과거 곱셈 수업의 에피소드를 참고하며, 곱셈 수업을 통해 얻었던 인사이트를 수집하여 대답할 수 있는지 확인합니다.

In [ ]:
query2 = """선생님, 정수의 나눗셈이 너무 어려워요. 어떻게 하면 이해를 잘 할 수 있을지 알려주세요."""

logger.info(f"\n{'='*80}")
logger.info(f"Test 2: Query for Reflection + Episode Knowledge")
logger.info(f"{'='*80}")
logger.info(f"Query: {query2}\n")

response2 = math_teacher_agent(query2)


### Test 3: Query for Specific Process Details
- 과거의 구체적인 에피소드를 떠올릴 수 있는지 확인합니다.
- 예를들어, 에피소드를 추출하는 과정에서 대화 내용을 지나치게 요약하거나 주요 내용을 삭제하여 저장하는지를 확인하기 위함입니다.

In [ ]:
query3 = """제가 지난번에 정수의 곱셈 문제 틀렸었는데, 그때 어떤 문제였고 어떻게 해결했었죠?"""

logger.info(f"\n{'='*80}")
logger.info(f"Query for Specific Process Details")
logger.info(f"{'='*80}")
logger.info(f"Query: {query3}\n")

response3 = math_teacher_agent(query3)


Episode 예시 :

```json
{
  "situation": "초등학교 수학 과외 첫 수업에서 교사가 학생 민수와 만나 두 자리 수와 한 자리 수의 곱셈, 특히 올림이 있는 세로셈을 가르치는 상황. 학생은 곱셈의 기본 개념은 알고 있지만 세로셈과 올림에 대해서는 처음 배우는 상태였음.",
  "intent": "1주차 수학 학습 목표인 두 자리 수와 한 자리 수의 곱셈에서 올림 개념을 정확히 가르치고, 학생이 이를 실제 문제에 적용할 수 있도록 하며, 동시에 학생 개인의 학습 습관(급하게 계산하는 문제) 개선까지 달성하고자 함.",
  "assessment": "Yes",
  "justification": "학생은 올림 개념을 완전히 이해했고(turn_10-11에서 23×4=92 정답), 급하게 계산하는 습관의 문제점을 인식하여 \"천천히 풀고 올림 확인\"하겠다고 다짐했으며(turn_13), 이후 15×6=90 문제에서 실제로 개선된 모습을 보였음(turn_15). 마지막에 핵심 내용인 올림의 중요성도 정확히 정리함(turn_16).",
  "reflection": "이 대화는 개념 학습과 학습 습관 개선을 동시에 달성한 성공적인 교육 사례를 보여줌. 효과적인 패턴으로는: 1) 구체적 예제를 통한 추상 개념 설명(12×3, 14×3으로 올림 도입), 2) 오답 발생 시 즉시 정답만 제시하지 않고 과정 확인을 통한 원인 진단, 3) 학생 스스로 문제점을 인식하게 하는 질문 활용(\"무엇이 빠졌는지\", \"앞으로 어떻게 할지\"), 4) 즉시 연습 기회 제공으로 개선 효과 확인. 특히 단순한 계산 실수를 학습 습관 문제로 확장하여 근본적 개선을 유도한 점이 뛰어남. 피해야 할 패턴으로는 추상적 설명만으로 개념을 가르치려 하거나, 오답에 대해 정답만 알려주고 넘어가는 것이 있음.",
  "turns": [
    {
      "situation": "학생이 23×4 문제에서 82라는 오답을 제시함",
      "intent": "오답임을 알리고 계산 과정을 확인하여 오류 원인 파악",
      "action": "오답임을 명시하고 계산 과정 설명을 요청하여 문제점 진단",
      "thought": "단순히 정답만 제시하지 않고 과정을 확인하여 구체적인 오류 지점을 찾아 교정 필요",
      "assessmentAssistant": "Yes - 오답 지적과 과정 확인 요청으로 오류 진단 시작",
      "assessmentUser": "No - 학생이 계산 과정을 설명하며 오류가 드러남"
    },
    "..."
  ]
}
```



### Test 4: Query for Pattern Recognition
- 이전 주차에서 진행한 수업들에 걸쳐 Insight을 잘 생성하여 Reflection Memory에 저장하였는지 확인해 봅니다.
- 예를들어, 1~3주차에 걸쳐 민수가 실수하는 경향, 문제를 틀리는 패턴 등에 대한 Insight을 저장하고 있는지 확인합니다.

In [ ]:
query4= """선생님, 제가 틀리는 문제들의 패턴이 있나요?"""

logger.info(f"\n{'='*80}")
logger.info(f"Query for Pattern Recognition")
logger.info(f"{'='*80}")
logger.info(f"Query: {query4}\n")

response4 = math_teacher_agent(query4)

## Step 8: (Optional) Direct Memory Inspection

직접적으로 Episode Memory와 Reflection Memory을 Retrieve 해봅니다. 

> 📌 참고 : Advanced 한 결과를 위해서 아래와 같은 Direct Inspection 방식을 이용하여 Retriever을 구성하고, 이를 Tool으로서 Agent에게 연동할 수 있습니다.

In [ ]:
# Inspect episodes directly using boto3
logger.info("" + "="*80)
logger.info("Direct Episode Inspection")
logger.info("="*80)

# Retrieve episodes using boto3 directly
namespace = f"math_learning/{actor_id}/sessions/" 
response = client.retrieve_memory_records(
    memoryId=memory_id,
    namespace=namespace,
    searchCriteria={
        'searchQuery': '정수의 나눗셈 수업',
        'topK': 1
    },
    maxResults=10
)

episodes = response.get('memoryRecordSummaries', [])

print(f"Found {len(episodes)} episodes in memory:")
for idx, episode in enumerate(episodes, 1):
    print(f"Episode {idx}:")
    pprint(episode, depth=2, width=100)
    print("-" * 80)

Episode Memory만을 필터하기 위해서는 `{"metadataValue": {"stringValue": "BASE"}}` 와 같이 필터를 설정하여 조회합니다. Reflection Memory을 조회하기 위해서는 `{"metadataValue": {"stringValue": "REFLECTION"}}`으로 설정합니다.

> 📌 참고 : Retrieve시에는 Episodes Memory Records는 "intent"을 기준으로 인덱스 되며, Reflection Memory Records들은 "use case"을 기준으로 인덱스 됩니다.

In [ ]:
import pprint

reflection_namespace = f"math_learning/{actor_id}" 

response = client.retrieve_memory_records(
memoryId=memory_id,
namespace=reflection_namespace,
searchCriteria={
    'searchQuery': "분수에 대한 이해도",
    "metadataFilters":[{
        "left":{"metadataKey": "x-amz-agentcore-memory-recordType"},
        "operator": "EQUALS_TO",
        "right": {"metadataValue": {"stringValue": "REFLECTION"}}
                        }],          
    'topK': 10
},
    maxResults=20
)

reflections = response.get('memoryRecordSummaries', [])
logger.info(f"   Found {len(reflections)} relevant reflections")
if reflections:
    for reflection in reflections:
        reflection_json = json.loads(reflection['content']['text'])
        pprint.pp(reflection_json)

## Cleanup (Optional)

Uncomment to delete the memory resource when done.

In [ ]:
# Uncomment to delete memory resource using boto3
# try:
#     memory_client.delete_memory(memoryId=memory_id, clientToken=str(uuid.uuid4()))
#     logger.info(f"✅ Successfully deleted memory: {memory_id}")
# except Exception as e:
#     logger.error(f"Error deleting memory: {e}")